This code aims to take a shapefile of station locations and measuements and removes a subset of stations at random to rerun in Greg's code. This could be implemented in some sort of potential cross validation.

In [ ]:
import sys
import os as os

import geopandas as gpd
import numpy as np
from numpy.polynomial import Polynomial
import psycopg2
from netCDF4 import Dataset

from tqdm import tqdm
from multiprocessing import Pool
import statsmodels.api as sm
from scipy import stats
# from scipy import optimize

import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature as cfNEF

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches
#import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

import rasterio
import xarray as xr

import random
import subprocess
import netCDF4
import shutil

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit

import pathlib
import glob
import pandas as pd

The following code looks at all the shapefiles within the assimilations and has an ouput of the number of each type of station. This is to help decide which assimilations are worth potentially researching.

In [ ]:
# Specify which folder to look through
fldr = 'C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024'

# Creates a list of file paths to shapefiles in the specified folder
files = [str(file) for file in pathlib.Path(fldr).rglob('*.shp')]

# List to store DataFrames for each shapefile
dfs = []

for file in files:
    # Read each shapefile
    gdf = gpd.read_file(file)
    
    # Check if 'STATION_TY' column exists in the GeoDataFrame
    if 'STATION_TY' in gdf.columns:
        # Count the occurrences of each station type
        station_type_counts = gdf['STATION_TY'].value_counts()
        
        # Convert the counts to a DataFrame
        df = pd.DataFrame(station_type_counts.items(), columns=['STATION_TY', 'Count'])
        
        # Add a column for the filename (matches with station counts)
        df['File'] = file
        
        # Append the DataFrame to the list
        dfs.append(df)
    else:
        print(f"File {file} does not contain 'STATION_TY' column.")

# Concatenate all DataFrames into a single DataFrame
result_df = pd.concat(dfs, ignore_index=True)

# Save the result DataFrame to a CSV file (This is saved within the src file)
result_df.to_csv('station_type_counts.csv', index=False)

In [ ]:
UMRB = result_df.loc[result_df['STATION_TY'].str.contains('UMRB')] # Only 36 files contain UMRB stations with most around only 1 staion and the highest being 15 stations

This code uses a shuffle split cross validator to remove test stations from shapefile and then runs an assimilation using IDW interpolation. The output files consist of a nuding layer image and a netCDF file showing the data from that nudging layer. They are saved within the src folder which can then be moved manually to a different folder of choice:

In [ ]:
# Inputs for code:
net_CDF1_Path = "C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/snodas_assim_20221214/ssm1054_2022121412.nc" 
net_CDF2_Path = "C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/snodas_assim_20221214/ssm_process_region_2022121312_2022121412_swe_west.nc"
net_CDF_name_us = "ssm_process_region_2022121312_2022121412_swe_west.nc"
net_CDF_name = "ssm1054_2022121412.nc"
folder_name = "snodas_assim_20221214"
shapefile_name = "ssm1054_md_based_2022121312_2022121412_west.shp"
start_date = "2022121312" # These are the numbers within the shapefile like shown in shapefile name above
end_date = "2022121412" # These are the numbers within the shapefile like shown in shapefile name above

# Load the shapefile:
shapefile_path = f"C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/{folder_name}/{shapefile_name}"
gdf = gpd.read_file(shapefile_path)

# Filter out rows rows with 'SNOTEL' in 'STATION_TY' switch this with UMRB Stations!!!!
gdf_filtered = gdf[gdf['STATION_TY'] == 'MESO-UMRB']

# Initialize ShuffleSplit
sss = ShuffleSplit(n_splits=1, test_size=0.10, random_state=0) # n_splits define how many test runs to do and test_size takes a percentage of gdf_filtered as test group

# Generate indices for splitting
for i, (train_index, test_index) in enumerate(sss.split(gdf_filtered)):  # Use gdf_filtered for generating test indices
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")
    
    # Extract training subset from gdf except the rows in test_subset
    train_subset = gdf.drop(test_index)
    
    # Extract test subset from gdf_filtered
    test_subset = gdf_filtered.iloc[test_index]
    
    # Print the size of each subset
    print(f"  Train subset size: {len(train_subset)}")
    print(f"  Test subset size: {len(test_subset)}")
    
    # Change folder and shapefile names
    folder_name = f"{folder_name}{i}"
    shapefile_name = f"ssm1054_md_based_2022121312_2022121412_west{i}.shp"
    net_CDF_name = f"ssm_process_region_2022121312_2022121412_swe_west{i}.nc"
    
    
    # Write new shapefile to new folder - had to save twice to get correct shapefile name - Ideas???
    train_subset.to_file(f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}")
    train_subset.to_file(f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{shapefile_name}")
    
        
    # Copy netCDF's from orgignal folder to new folder (Need original netCDF's to run)
    shutil.copy(net_CDF1_Path,f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{net_CDF_name}" )
    shutil.copy(net_CDF2_Path,f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{net_CDF_name_us}" )
    
    # Enter command you want to run (Look at commands in Greg's Examples, this command uses assigned IDW creteria to create new nudging layer) 
    cmd = f"python ./snodas_idw.py -k -i 3 50 1 0 250 {start_date} {end_date} west{i} -f ../Test_Runs/{folder_name}" #Careful with the US, West, etc. part in command
    
    # Run Greg's code using the above inputs and chosen command in cmd line
    subprocess.run(cmd, capture_output=True)

Notes for Paul:

The above code uses a shuffle split cross validator to remove a test group of stations from the shapefile. It is able to take any assimilation file that we downloaded and with updataing the input field should be able to run a new assimilation on the updated shapefile with the test stations removed. A new net_CDF output file and nudging image for the run is saved in the source folder within the Repo. The newly created shapefiles that are created for this run are saved in their own folders within UMRB-SNODAS Repo called Test_Runs. 
